# LambdaRankIC Pairwise Rank IC PIT Grid

This notebook turns `docs/research/current/LOSS_PATH_DECISION_2026-06-04.md`
into a Colab-ready PIT comparison. It keeps the frozen pure-IC launch
candidate in the grid, includes the Portfolio-IC hybrid as the current
portfolio-utility branch, and adds the disabled-by-default
`lambdarank_ic` candidate.

Recipe reference: `docs/DEFAULT_EXPERIMENT_RECIPE.md`.

Default mode is a lower-pair screen: one year, one seed, one model,
40 epochs, patience 8, and LambdaRankIC pair caps
`[512, 1024, 2048, 4096]`. This is intentionally cheaper than the full
20-model ensemble and should be labeled as screening evidence.

Operate this notebook from the visible Colab UI on a G4/L4-class Colab
runtime, not T4/CPU. Use Drive artifacts as truth if output streaming
stalls; Drive API fallback is preferred over repeated DriveFS remounts
for compact artifacts such as heartbeat and result files. If cleanup
does not run, use `Runtime > Disconnect and delete runtime` manually.

## 1. Setup

In [ ]:
import csv
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import torch

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/magilliam27/MCI-GRU.git"
BRANCH = "codex/lambdarankic-lower-pair-screen"
REPO_DIR = Path("/content/MCI-GRU") if IN_COLAB else Path.cwd()
REQUIRE_G4_L4_GPU = True
BLOCKED_GPU_NAMES = ("T4",)

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[dev,tracking,fred]"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repo:", REPO_DIR)
print("Branch:", BRANCH)
subprocess.run(["git", "rev-parse", "HEAD"], check=False)
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if REQUIRE_G4_L4_GPU and any(blocked in gpu_name for blocked in BLOCKED_GPU_NAMES):
        raise RuntimeError(
            f"Expected G4/L4-class Colab runtime, not T4/CPU. Visible GPU: {gpu_name}"
        )
elif REQUIRE_G4_L4_GPU:
    raise RuntimeError(
        "Expected G4/L4-class Colab runtime, not T4/CPU. "
        "Switch Runtime -> Change runtime type -> G4 GPU before training."
    )

from mci_gru.config import TrainingConfig
from mci_gru.training.losses import build_training_loss

probe_cfg = TrainingConfig(loss_type="lambdarank_ic", selection_metric="val_rank_ic")
probe_loss, probe_name = build_training_loss(probe_cfg)
print("LambdaRankIC branch probe:", probe_name, type(probe_loss).__name__)

## 2. FRED Key And PIT Data

In [ ]:
if IN_COLAB and not os.environ.get("FRED_API_KEY"):
    try:
        from google.colab import userdata

        secret = userdata.get("FRED_API_KEY")
        if secret:
            os.environ["FRED_API_KEY"] = secret
            print("FRED_API_KEY loaded from Colab Secrets.")
    except Exception as exc:
        print("Could not read FRED_API_KEY from Colab Secrets:", exc)

if not os.environ.get("FRED_API_KEY"):
    raise RuntimeError("FRED_API_KEY is required for the current regime-enabled preset.")

drive_data_dir = Path("/content/drive/MyDrive/MCI_GRU_shared/data") if IN_COLAB else REPO_DIR / "data/raw/market"
drive_market_csv = drive_data_dir / "sp500_pit_union_lseg_20150101_20260513.csv"
drive_pit_csv = drive_data_dir / "sp500_pit_joiner_leaver_20160101_20260513_pit_universe.csv"

if not drive_market_csv.exists():
    raise FileNotFoundError(f"Missing market CSV: {drive_market_csv}")
if not drive_pit_csv.exists():
    raise FileNotFoundError(f"Missing PIT universe CSV: {drive_pit_csv}")

repo_market_csv = REPO_DIR / "data/raw/market/sp500_pit_union_lseg_20150101_20260513.csv"
repo_pit_csv = REPO_DIR / "data/raw/constituents/sp500_pit_joiner_leaver_20160101_20260513_pit_universe.csv"
repo_market_csv.parent.mkdir(parents=True, exist_ok=True)
repo_pit_csv.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(drive_market_csv, repo_market_csv)
shutil.copy2(drive_pit_csv, repo_pit_csv)

print("Market CSV:", repo_market_csv)
print("PIT CSV:", repo_pit_csv)

## 3. Build Objective Grid

In [ ]:
SMOKE_MODE = False
SCREEN_MODE = True
RUN_TRAINING = True
MAX_JOBS = None
SMOKE_YEARS = [2022]
SCREEN_YEARS = [2022]
FULL_YEARS = [2022, 2023, 2024, 2025]
SMOKE_BASE_SEEDS = [314159]
SCREEN_BASE_SEEDS = [314159]
FULL_BASE_SEEDS = [314159, 271828, 161803]
SMOKE_PAIR_CAPS = [512]
SCREEN_PAIR_CAPS = [512, 1024, 2048, 4096]
FULL_PAIR_CAPS = [4096]
SMOKE_NUM_MODELS = 1
SCREEN_NUM_MODELS = 1
FULL_NUM_MODELS = 20
SMOKE_NUM_EPOCHS = 2
SCREEN_NUM_EPOCHS = 40
FULL_NUM_EPOCHS = 100
SMOKE_EARLY_STOPPING_PATIENCE = 2
SCREEN_EARLY_STOPPING_PATIENCE = 8
FULL_EARLY_STOPPING_PATIENCE = 15

if SMOKE_MODE and SCREEN_MODE:
    raise ValueError("SMOKE_MODE and SCREEN_MODE are mutually exclusive.")

BUDGET_MODE = "smoke" if SMOKE_MODE else ("screen" if SCREEN_MODE else "full")
YEARS = SMOKE_YEARS if SMOKE_MODE else (SCREEN_YEARS if SCREEN_MODE else FULL_YEARS)
BASE_SEEDS = (
    SMOKE_BASE_SEEDS
    if SMOKE_MODE
    else (SCREEN_BASE_SEEDS if SCREEN_MODE else FULL_BASE_SEEDS)
)
PAIR_CAPS = SCREEN_PAIR_CAPS if SCREEN_MODE else (SMOKE_PAIR_CAPS if SMOKE_MODE else FULL_PAIR_CAPS)
NUM_MODELS = (
    SMOKE_NUM_MODELS
    if SMOKE_MODE
    else (SCREEN_NUM_MODELS if SCREEN_MODE else FULL_NUM_MODELS)
)
NUM_EPOCHS = (
    SMOKE_NUM_EPOCHS
    if SMOKE_MODE
    else (SCREEN_NUM_EPOCHS if SCREEN_MODE else FULL_NUM_EPOCHS)
)
EARLY_STOPPING_PATIENCE = (
    SMOKE_EARLY_STOPPING_PATIENCE
    if SMOKE_MODE
    else (
        SCREEN_EARLY_STOPPING_PATIENCE
        if SCREEN_MODE
        else FULL_EARLY_STOPPING_PATIENCE
    )
)

RUN_TAG = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = (
    Path("/content/drive/MyDrive/MCI-GRU-Ablations/lambdarank_ic_pit")
    if IN_COLAB
    else REPO_DIR / "results" / "lambdarank_ic_pit"
) / RUN_TAG
TRAINING_OUTPUT_DIR = RUN_ROOT / "training"
HEARTBEAT_PATH = RUN_ROOT / "heartbeat.json"
RESULTS_CSV_PATH = RUN_ROOT / "training_results.csv"
RESULTS_JSON_PATH = RUN_ROOT / "training_results.json"
LEGACY_RESULTS_JSON_PATH = RUN_ROOT / "lambdarank_ic_pit_training_results.json"
if RUN_ROOT.exists():
    raise RuntimeError(f"Refusing to reuse existing run root: {RUN_ROOT}")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def write_heartbeat(
    phase: str,
    status: str = "RUNNING",
    current_job: str | None = None,
    completed_jobs: int = 0,
    error: str | None = None,
) -> None:
    payload = {
        "phase": phase,
        "status": status,
        "current_job": current_job,
        "completed_jobs": completed_jobs,
        "expected_jobs": None,
        "budget_mode": BUDGET_MODE,
        "branch": BRANCH,
        "run_root": str(RUN_ROOT),
        "updated_at": datetime.utcnow().isoformat() + "Z",
    }
    if "jobs" in globals():
        payload["expected_jobs"] = len(jobs)
    if torch.cuda.is_available():
        payload["gpu"] = torch.cuda.get_device_name(0)
    if error is not None:
        payload["error"] = error
    HEARTBEAT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")

FROZEN_RECIPE_ID = (
    "static-threshold-shuffle__pure-ic-returns-5d-val-ic__"
    "regime-current-only__ensemble__drop-edge-0p1"
)

OBJECTIVE_VARIANTS_FULL = {
    'pure_ic_baseline': {
        'loss_type': 'ic',
        'selection_metric': 'val_ic',
    },
    'portfolio_ic_hybrid': {
        'loss_type': 'portfolio_ic',
        'selection_metric': 'val_loss',
        'portfolio_ic_top_k': 10,
        'portfolio_ic_weight': 0.25,
        'portfolio_ic_temperature': 0.25,
    },
    'lambdarank_ic_candidate': {
        'loss_type': 'lambdarank_ic',
        'selection_metric': 'val_rank_ic',
        'lambdarank_ic_max_pairs_per_day': 4096,
        'lambdarank_ic_temperature': 1.0,
    },
}
OBJECTIVE_VARIANTS = (
    {
        'lambdarank_ic_pair_cap_screen': {
            'loss_type': 'lambdarank_ic',
            'selection_metric': 'val_rank_ic',
            'lambdarank_ic_max_pairs_per_day': 4096,
            'lambdarank_ic_temperature': 1.0,
        },
    }
    if SCREEN_MODE
    else OBJECTIVE_VARIANTS_FULL
)

pair_multiplier = sum(
    len(PAIR_CAPS) if variant["loss_type"] == "lambdarank_ic" else 1
    for variant in OBJECTIVE_VARIANTS.values()
)
EXPECTED_JOB_COUNT = len(YEARS) * len(BASE_SEEDS) * pair_multiplier
EXPECTED_TOTAL_MODELS = EXPECTED_JOB_COUNT * NUM_MODELS
expected_jobs_by_mode = len(SCREEN_PAIR_CAPS) if SCREEN_MODE else (3 if SMOKE_MODE else 36)
expected_models_by_mode = len(SCREEN_PAIR_CAPS) if SCREEN_MODE else (3 if SMOKE_MODE else 720)
assert EXPECTED_JOB_COUNT == expected_jobs_by_mode
assert EXPECTED_TOTAL_MODELS == expected_models_by_mode

BASE_OVERRIDES = [
    "data.source=csv",
    "features=with_momentum",
    "features.include_momentum=true",
    "features.include_weekly_momentum=true",
    "features.momentum_encoding=binary",
    "features.momentum_blend_mode=static",
    "features.momentum_blend_fast_weight=0.5",
    "features.include_global_regime=true",
    "features.regime_strict=true",
    "features.regime_enforce_lag_days=0",
    "features.regime_include_subsequent_returns=false",
    "features.regime_change_months=12",
    "features.regime_norm_months=120",
    "features.regime_exclusion_months=1",
    "features.regime_similarity_quantile=0.2",
    "features.regime_min_history_months=24",
    "graph.judge_value=0.8",
    "graph.update_frequency_months=0",
    "graph.corr_lookback_days=252",
    "graph.top_k=0",
    "graph.top_k_metric=corr",
    "graph.use_multi_feature_edges=true",
    "graph.append_snapshot_age_days=false",
    "graph.use_lead_lag_features=false",
    "graph.drop_edge_p=0.1",
    "training.lr_scheduler=cosine",
    "training.learning_rate=5e-5",
    f"training.num_epochs={NUM_EPOCHS}",
    f"training.num_models={NUM_MODELS}",
    f"training.early_stopping_patience={EARLY_STOPPING_PATIENCE}",
    "training.label_type=returns",
    "training.shuffle_train=true",
    "model.label_t=5",
    "model.temporal_encoder=gru_attn",
    "tracking.enabled=false",
    "tracking.log_artifacts=false",
    "tracking.log_checkpoints=false",
    "tracking.log_predictions=false",
    f"data.filename={repo_market_csv.relative_to(REPO_DIR).as_posix()}",
    f"data.pit_universe_csv={repo_pit_csv.relative_to(REPO_DIR).as_posix()}",
    "data.use_pit_universe=true",
    "data.pit_universe_mode=masked_panel",
    "data.pit_min_scoreable_stocks=450",
    "data.pit_breadth_policy=error",
]

def loss_overrides_for_variant(variant: dict) -> list[str]:
    overrides = [
        f"training.loss_type={variant['loss_type']}",
        f"training.selection_metric={variant['selection_metric']}",
    ]
    if variant["loss_type"] == "portfolio_ic":
        overrides.extend(
            [
                f"training.portfolio_ic_top_k={variant['portfolio_ic_top_k']}",
                f"training.portfolio_ic_weight={variant['portfolio_ic_weight']}",
                f"training.portfolio_ic_temperature={variant['portfolio_ic_temperature']}",
            ]
        )
    if variant["loss_type"] == "lambdarank_ic":
        overrides.extend(
            [
                "training.lambdarank_ic_max_pairs_per_day="
                f"{variant['lambdarank_ic_max_pairs_per_day']}",
                "training.lambdarank_ic_temperature="
                f"{variant['lambdarank_ic_temperature']}",
            ]
        )
    return overrides

jobs = []
for year in YEARS:
    for base_seed in BASE_SEEDS:
        for variant_name, variant in OBJECTIVE_VARIANTS.items():
            variant_pair_caps = PAIR_CAPS if variant["loss_type"] == "lambdarank_ic" else [None]
            for max_pairs_per_day in variant_pair_caps:
                experiment = f"pit_temporal_{year}"
                job_variant = dict(variant)
                if max_pairs_per_day is not None:
                    job_variant["lambdarank_ic_max_pairs_per_day"] = max_pairs_per_day
                pair_suffix = (
                    f"_pairs{max_pairs_per_day}" if max_pairs_per_day is not None else ""
                )
                name = f"lambdarank_ic_{variant_name}{pair_suffix}_{year}_seed{base_seed}"
                jobs.append(
                    {
                        "year": year,
                        "base_seed": base_seed,
                        "variant": variant_name,
                        "loss_type": job_variant["loss_type"],
                        "selection_metric": job_variant["selection_metric"],
                        "max_pairs_per_day": max_pairs_per_day,
                        "name": name,
                        "overrides": [
                            f"+experiment={experiment}",
                            *BASE_OVERRIDES,
                            *loss_overrides_for_variant(job_variant),
                            f"seed={base_seed}",
                            f"experiment_name={name}",
                            f"output_dir={TRAINING_OUTPUT_DIR.as_posix()}",
                        ],
                    }
                )

if MAX_JOBS is not None:
    jobs = jobs[:MAX_JOBS]

manifest = {
    "research_basis": "LOSS_PATH_DECISION_2026-06-04.md",
    "recipe_id": FROZEN_RECIPE_ID,
    "branch": BRANCH,
    "run_tag": RUN_TAG,
    "smoke_mode": SMOKE_MODE,
    "screen_mode": SCREEN_MODE,
    "budget_mode": BUDGET_MODE,
    "years": YEARS,
    "base_seeds": BASE_SEEDS,
    "pair_caps": PAIR_CAPS,
    "num_models": NUM_MODELS,
    "num_epochs": NUM_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "expected_job_count": EXPECTED_JOB_COUNT,
    "expected_total_models": EXPECTED_TOTAL_MODELS,
    "objective_variants": OBJECTIVE_VARIANTS,
    "jobs": jobs,
}
manifest_path = RUN_ROOT / "lambdarank_ic_pit_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Run root:", RUN_ROOT)
print("Recipe:", FROZEN_RECIPE_ID)
print("Jobs:", len(jobs))
print("Expected total models:", EXPECTED_TOTAL_MODELS)
for job in jobs:
    print("-", job["name"], job["loss_type"], job["selection_metric"])
print("Manifest:", manifest_path)
write_heartbeat("manifest", completed_jobs=0)

## 4. Run Training Jobs

In [ ]:
def latest_training_summary(job_name: str) -> Path | None:
    candidates = sorted(TRAINING_OUTPUT_DIR.glob(f"{job_name}/**/training_summary.json"))
    return candidates[-1] if candidates else None

def write_results_artifacts(rows: list[dict]) -> None:
    RESULTS_JSON_PATH.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    LEGACY_RESULTS_JSON_PATH.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    fieldnames = [
        "name",
        "variant",
        "loss_type",
        "selection_metric",
        "year",
        "base_seed",
        "max_pairs_per_day",
        "returncode",
        "elapsed_seconds",
        "training_summary_path",
    ]
    with RESULTS_CSV_PATH.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key) for key in fieldnames})

results = []
try:
    if not RUN_TRAINING:
        print("RUN_TRAINING is false; grid manifest only.")
        write_heartbeat("skipped", status="OK", completed_jobs=0)
    for job in jobs if RUN_TRAINING else []:
        write_heartbeat(
            "training",
            current_job=job["name"],
            completed_jobs=len(results),
        )
        print("=" * 100)
        print("Starting:", job["name"])
        cmd = [sys.executable, "-u", str(REPO_DIR / "run_experiment.py"), *job["overrides"]]
        print("Command:", " ".join(cmd[:4]), "... +", len(job["overrides"]), "overrides")
        env = {**os.environ, "PYTHONUNBUFFERED": "1"}
        start_time = time.perf_counter()
        proc = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, env=env)
        elapsed_seconds = time.perf_counter() - start_time
        summary_path = latest_training_summary(job["name"])
        result = {
            "name": job["name"],
            "variant": job["variant"],
            "loss_type": job["loss_type"],
            "selection_metric": job["selection_metric"],
            "year": job["year"],
            "base_seed": job["base_seed"],
            "max_pairs_per_day": job["max_pairs_per_day"],
            "returncode": int(proc.returncode),
            "elapsed_seconds": round(elapsed_seconds, 3),
            "training_summary_path": str(summary_path) if summary_path else None,
        }
        if summary_path is not None:
            result["training_summary"] = json.loads(summary_path.read_text(encoding="utf-8"))
        results.append(result)
        write_results_artifacts(results)
        print("Return code:", proc.returncode)
        print("Training summary:", summary_path)
        if proc.returncode != 0:
            raise RuntimeError(f"Job failed: {job['name']}")

    print("Training loop complete.")
    write_heartbeat("done", status="OK", completed_jobs=len(results))
except Exception as exc:
    write_heartbeat(
        "failed",
        status="FAILED",
        current_job=job["name"] if "job" in locals() else None,
        completed_jobs=len(results),
        error=str(exc),
    )
    raise
finally:
    if IN_COLAB:
        try:
            import google.colab.runtime

            google.colab.runtime.unassign()
            print("Released Colab runtime with google.colab.runtime.unassign().")
        except Exception as exc:
            print(
                "Runtime > Disconnect and delete runtime manually if "
                f"foreground cleanup did not complete: {exc}"
            )

print("Results:", RESULTS_JSON_PATH)
print("Legacy results:", LEGACY_RESULTS_JSON_PATH)
print("CSV results:", RESULTS_CSV_PATH)

## 5. Results Snapshot

In [ ]:
results_path = RESULTS_JSON_PATH
if results_path.exists():
    import pandas as pd

    df = pd.json_normalize(json.loads(results_path.read_text(encoding="utf-8")))
    keep_cols = [
        "variant",
        "loss_type",
        "selection_metric",
        "year",
        "base_seed",
        "returncode",
        "training_summary.mean_best_val_ic",
        "training_summary.mean_best_val_rank_ic",
        "training_summary.evaluation.avg_ic",
        "training_summary.evaluation.avg_rank_ic",
    ]
    display(df[[col for col in keep_cols if col in df.columns]])
else:
    print("No training results file yet.")

print("Primary go/no-go question:")
print("Does lambdarank_ic improve Rank IC and net top-k/rank-drop backtests versus pure IC without increasing turnover, drawdown, or year instability?")